In [112]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.pyplot import colorbar
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [113]:
df = pd.read_csv('../data/security_access_risk_dataset_RNC.csv')
df.head()

,Failed_Attempts_Last30Days,Access_Time_Deviation,Entry_Distance_From_Usual,User_Access_Tier,Access_Risk_Score
0,3.0,16.30,41.03,Standard,68.10
1,0.0,13.95,31.86,Admin,43.55
2,4.0,30.83,36.13,Standard,85.70
3,3.0,17.50,51.10,Standard,66.51
4,3.0,11.93,25.94,Standard,46.72


In [114]:
print(df.columns)
print(df.shape)
print(df.dtypes)
df.describe()

Index(['Failed_Attempts_Last30Days', 'Access_Time_Deviation',
       'Entry_Distance_From_Usual', 'User_Access_Tier', 'Access_Risk_Score'],
      dtype='object')
(1545, 5)
Failed_Attempts_Last30Days    float64
Access_Time_Deviation         float64
Entry_Distance_From_Usual     float64
User_Access_Tier               object
Access_Risk_Score             float64
dtype: object


,Failed_Attempts_Last30Days,Access_Time_Deviation,Entry_Distance_From_Usual,Access_Risk_Score
count,1467.00000,1468.000000,1466.000000,1467.000000
mean,1.98773,15.578569,30.469836,52.975235
std,1.40205,8.904478,18.550608,20.714527
min,0.00000,0.040000,0.010000,0.010000
25%,1.00000,8.900000,15.885000,38.660000
50%,2.00000,14.710000,29.245000,51.720000
75%,3.00000,21.677500,42.937500,65.815000
max,8.00000,45.150000,94.800000,126.210000


In [115]:
df.isnull().sum()

Failed_Attempts_Last30Days    78
Access_Time_Deviation         77
Entry_Distance_From_Usual     79
User_Access_Tier              77
Access_Risk_Score             78
dtype: int64

In [116]:
df = df.fillna(df.mean(numeric_only=True))
df.fillna({'User_Access_Tier': 'Standard'}, inplace = True)
print(df.dtypes)
df.isnull().sum()

Failed_Attempts_Last30Days    float64
Access_Time_Deviation         float64
Entry_Distance_From_Usual     float64
User_Access_Tier               object
Access_Risk_Score             float64
dtype: object


Failed_Attempts_Last30Days    0
Access_Time_Deviation         0
Entry_Distance_From_Usual     0
User_Access_Tier              0
Access_Risk_Score             0
dtype: int64

In [117]:
df["User_Access_Tier"] = df["User_Access_Tier"].map({'Standard':1 , 'Elevated':2 , 'Admin':3})
df.head()

,Failed_Attempts_Last30Days,Access_Time_Deviation,Entry_Distance_From_Usual,User_Access_Tier,Access_Risk_Score
0,3.0,16.30,41.03,1,68.10
1,0.0,13.95,31.86,3,43.55
2,4.0,30.83,36.13,1,85.70
3,3.0,17.50,51.10,1,66.51
4,3.0,11.93,25.94,1,46.72


In [129]:
print(df.dtypes)

Y = df['Access_Risk_Score']
X = df.drop(columns=['Access_Risk_Score'])

X_train,Y_train, X_test, Y_test = train_test_split(X, Y,test_size=0.2,random_state=42)

Failed_Attempts_Last30Days    float64
Access_Time_Deviation         float64
Entry_Distance_From_Usual     float64
User_Access_Tier                int64
Access_Risk_Score             float64
dtype: object


In [135]:
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(np.array(X_train))
print(normalizer.mean.numpy())

[[ 1.99456   15.607169  30.563553   1.4401294]]


In [120]:
linear_model = tf.keras.Sequential([
    normalizer,
    layers.Dense(units=1)
])

In [121]:
linear_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss='mean_absolute_error')